In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../kaggle_prediction_library/') 
import preprocess
import feature_engineering
import submission
from sklearn.model_selection import train_test_split

# from hyperopt import tpe, fmin, Trials
# import hyperopt.hp as hp


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import log_loss
from sklearn.feature_selection import chi2
from sklearn.metrics import r2_score
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import brier_score_loss


### Read data

In [3]:
regular_season_results = pd.read_csv('../data/MRegularSeasonDetailedResults.csv')
detailed_tourney_results = pd.read_csv('../data/MNCAATourneyDetailedResults.csv')
rankings = pd.read_csv('../data/MMasseyOrdinals.csv')
seeds = pd.read_csv('../data/MNCAATourneySeeds.csv')

# kp_rankings = pd.read_csv('../data/kenpom_pre_tourney_snapshot.csv')

regular_season_results_w = pd.read_csv('../data/WRegularSeasonDetailedResults.csv')
detailed_tourney_results_w = pd.read_csv('../data/WNCAATourneyDetailedResults.csv')

mteams = pd.read_csv('../data/MTeams.csv')
wteams = pd.read_csv('../data/WTeams.csv')

seeds_w = pd.read_csv('../data/WNCAATourneySeeds.csv')

# M538 = pd.read_csv('../data/M538.csv')
# W538 = pd.read_csv('../data/W538.csv')

seed_round = pd.read_csv("../data/MNCAATourneySeedRoundSlots.csv")
seeds = pd.read_csv("../data/MNCAATourneySeeds.csv")

first_round_odds_data = pd.read_csv('../data/sky_data/first_round_odds_ncaam.csv')
first_round_odds_data_w = pd.read_csv('../data/sky_data/first_round_odds_ncaaw.csv')


In [4]:
torvik_player_data = pd.read_csv("../data/sky_data/torvik_player_data_2008_2024.csv")
torvik_player_data_ncaaw = pd.read_csv("../data/sky_data/ncaaw_torvik_player_data_2021_2024.csv")


In [5]:
#aggregated_player_stats = pd.read_csv("../data/sky_data/aggregated_player_stats.csv")

In [6]:
sub_df = pd.read_csv("SampleSubmission2024.csv")

### Set Up Data

In [7]:
to_predict_mens, to_predict_womens, regular_season_games, regular_season_games_w = preprocess.full_setup(detailed_tourney_results, regular_season_results,
               detailed_tourney_results_w, regular_season_results_w,
               sub_df, mteams)

/Users/skylerdale/workspace/kaggle-ncaam/2025_model/development_notebooks/../kaggle_prediction_library/submission.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat([mens_historical_games, womens_historical_games, sub], axis = 0)


### Add Features

In [8]:
to_predict_womens = feature_engineering.TournamentSeed(tourney_seeds=seeds_w).add(to_predict_womens)
to_predict_womens = feature_engineering.Efficiency(games=regular_season_games_w, away_bonus=0).add(to_predict_womens)
to_predict_womens = feature_engineering.RoundNumber(seeds, seed_round).add(to_predict_womens)
to_predict_womens = feature_engineering.TeamNames(wteams).add(to_predict_womens)

# to_predict_womens = feature_engineering.FiveThirtyEight(fivethirtyeight_df=W538).add(to_predict_womens)

/Users/skylerdale/workspace/kaggle-ncaam/2025_model/development_notebooks/../kaggle_prediction_library/feature_engineering.py:132: FutureWarning: The provided callable <function mean at 0x10542c0d0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  final = all_games3.groupby(['Season', 'Team1']).agg(adj_oe=('adj_oe', np.mean), adj_de=('adj_de', np.mean)).reset_index()


In [9]:
to_predict_womens = feature_engineering.FirstRoundOdds(first_round_odds_data_w).add(to_predict_womens)

In [10]:
# to remove later
to_predict_womens = to_predict_womens[(to_predict_womens.type != "Prediction")].copy()

In [11]:
to_predict_womens = feature_engineering.RoundNumber(seeds, seed_round).add(to_predict_womens)
to_predict_womens = feature_engineering.AggregatedPlayerStats(torvik_player_data_ncaaw).add(to_predict_womens)

In [12]:
#to_predict_womens.to_csv("../development_notebooks/to_predict_women.csv")
to_predict_womens.to_csv("to_predict_women.csv")

In [13]:
to_predict_mens = feature_engineering.TeamNames(mteams).add(to_predict_mens)
to_predict_mens = feature_engineering.FirstRoundOdds(first_round_odds_data).add(to_predict_mens)
to_predict_mens = feature_engineering.RoundNumber(seeds, seed_round).add(to_predict_mens)
to_predict_mens = feature_engineering.SeasonStats(regular_season_games).add(to_predict_mens)
# to_predict_mens = feature_engineering.FiveThirtyEight(fivethirtyeight_df=M538).add(to_predict_mens)
to_predict_mens = feature_engineering.PreSeasonAPRankings(rankings_df=rankings).add(to_predict_mens)
to_predict_mens = feature_engineering.TournamentSeed(tourney_seeds=seeds).add(to_predict_mens)
to_predict_mens = feature_engineering.Efficiency(games=regular_season_games, away_bonus=0).add(to_predict_mens)
to_predict_mens = feature_engineering.FinalRanking(rankings_df=rankings, system='WLK').add(to_predict_mens) # switched in 2024 because SAG dissapeared
# to_predict_mens = feature_engineering.Kenpom(kp_snapshot=kp_rankings).add(to_predict_mens)
# this one takes 3 minutes to run
# to_predict_mens = feature_engineering.TeamQuality(games=regular_season_games).add(to_predict_mens)


/Users/skylerdale/workspace/kaggle-ncaam/2025_model/development_notebooks/../kaggle_prediction_library/feature_engineering.py:223: FutureWarning: The provided callable <function mean at 0x10542c0d0> is currently using DataFrameGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  season_statistics = df.groupby(["Season", 'Team1'])[boxscore_cols].agg(np.mean).reset_index()
/Users/skylerdale/workspace/kaggle-ncaam/2025_model/development_notebooks/../kaggle_prediction_library/feature_engineering.py:132: FutureWarning: The provided callable <function mean at 0x10542c0d0> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  final = all_games3.groupby(['Season', 'Team1']).agg(adj_oe=('adj_oe', np.mean), adj_de=('adj_de', np.mean)).reset_index()
/Users/skylerdale/workspace/kaggle-

In [14]:
to_predict_mens = feature_engineering.AggregatedPlayerStats(torvik_player_data).add(to_predict_mens)


In [15]:
to_predict_mens[
    (to_predict_mens.final_odds.isnull())].groupby("Season").count()

# why 2024 mising?

,type,ID,Pred,Team1,Team2,Outcome,Gender,margin,t1_TeamName,t1_FirstD1Season,...,t1_top3_USG_gini,t1_top8_BPM_weighted_mean,t2_top8_TO_stdev,t2_top5_PRPG!_median,t2_top3_DR_median,t2_top5_STL_cv,t2_top3_Min%_median,t2_top8_TS_gini,t2_top3_USG_gini,t2_top8_BPM_weighted_mean
Season,,,,,,,,,,,,,,,,,,,,,
2006,2,2,0,2,2,2,2,2,2,2,...,0,0,0,0,0,0,0,0,0,0
2007,2,2,0,2,2,2,2,2,2,2,...,0,0,0,0,0,0,0,0,0,0
2016,2,2,0,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
2018,2,2,0,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
2021,2,2,0,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
2022,4,4,0,4,4,4,4,4,4,4,...,3,3,3,3,3,3,3,3,3,3
2024,506,506,496,506,506,10,506,10,506,506,...,506,506,506,506,506,506,506,506,506,506


In [16]:
# Original 

# to_predict_mens = to_predict_mens[(to_predict_mens.type != "Prediction") & 
#                                   (to_predict_mens.final_odds.notnull())
#                                   ]

# New - Not Prediction and (game round != 1 OR final odds is null)

to_predict_mens = to_predict_mens[(to_predict_mens.type != "Prediction") & 
                                  
                                  ( (to_predict_mens.final_odds.notnull()) | (to_predict_mens.GameRound != 1) )
                               
                                  ]

### Split Dataset

In [17]:
first_round_df = to_predict_mens[to_predict_mens.GameRound == 1].copy()
other_rounds_df = to_predict_mens[to_predict_mens.GameRound > 1].copy()

In [18]:
# first_round_df.to_csv("to_predict_mens_first_round.csv")
# other_rounds_df.to_csv("to_predict_mens_other_rounds.csv")
to_predict_mens.to_csv("to_predict_mens.csv")

In [19]:
list(to_predict_mens.columns)

['type',
 'ID',
 'Pred',
 'Season',
 'Team1',
 'Team2',
 'Outcome',
 'Gender',
 'margin',
 't1_TeamName',
 't1_FirstD1Season',
 't1_LastD1Season',
 't2_TeamName',
 't2_FirstD1Season',
 't2_LastD1Season',
 'final_odds',
 'GameRound',
 't1_FGM',
 't1_FGA',
 't1_FGM3',
 't1_FGA3',
 't1_OR',
 't1_Ast',
 't1_TO',
 't1_Stl',
 't1_PF',
 't1_FTA',
 't1_FTM',
 't1_PointDiff',
 't2_FGM',
 't2_FGA',
 't2_FGM3',
 't2_FGA3',
 't2_OR',
 't2_Ast',
 't2_TO',
 't2_Stl',
 't2_PF',
 't2_FTA',
 't2_FTM',
 't2_PointDiff',
 't1_OrdinalRank',
 't2_OrdinalRank',
 't1_Seed',
 't2_Seed',
 'seed_diff',
 't1_adj_oe',
 't1_adj_de',
 't1_adj_margin',
 't2_adj_oe',
 't2_adj_de',
 't2_adj_margin',
 't1_final_rank',
 't2_final_rank',
 't1_top8_TO_stdev',
 't1_top5_PRPG!_median',
 't1_top3_DR_median',
 't1_top5_STL_cv',
 't1_top3_Min%_median',
 't1_top8_TS_gini',
 't1_top3_USG_gini',
 't1_top8_BPM_weighted_mean',
 't2_top8_TO_stdev',
 't2_top5_PRPG!_median',
 't2_top3_DR_median',
 't2_top5_STL_cv',
 't2_top3_Min%_media